# Базовые модели и валидация

Сравниваем наивный прогноз и первую текстовую модель на одной отложенной выборке. Метрика — MAE: чем меньше, тем лучше. Исходный train.csv не меняем, а Kaggle test.csv здесь не используем, поскольку в нём нет оценок.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
data_path = root / "data" / "raw" / "train.csv"
if not data_path.exists():
    raise FileNotFoundError("Поместите train.csv в data/raw/")
train = pd.read_csv(data_path)
assert list(train.columns) == ["Id", "Review", "Rating"]
assert train["Review"].notna().all()
assert train["Rating"].isin(range(1, 6)).all()
print(f"Прочитано строк: {len(train):,}")

Прочитано строк: 60,000


## Очистка и разбиение

Удаляем точные повторы текста из рабочей копии. После этого группируем отзывы, которые различаются только регистром или пробелами: такие варианты не должны оказаться одновременно в обучении и валидации. Разбиение фиксируем заранее; примерно 20% строк попадёт в валидацию.

In [2]:
train_clean = train.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)
removed = len(train) - len(train_clean)
groups = train_clean["Review"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
print(f"Удалено точных повторов текста: {removed}")
print(f"Осталось строк: {len(train_clean):,}")
print(f"Дополнительных совпадений после нормализации: {groups.duplicated().sum()}")

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
train_idx, valid_idx = next(splitter.split(train_clean["Review"], train_clean["Rating"], groups))
train_part = train_clean.iloc[train_idx]
valid_part = train_clean.iloc[valid_idx]
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[valid_idx]))
assert set(train_part["Review"]).isdisjoint(set(valid_part["Review"]))
print(f"Обучение: {len(train_part):,}; валидация: {len(valid_part):,}")
print("Пересечений нормализованных текстов: 0")
class_share = pd.DataFrame({
    "обучение, %": 100 * train_part["Rating"].value_counts(normalize=True).sort_index(),
    "валидация, %": 100 * valid_part["Rating"].value_counts(normalize=True).sort_index(),
})
display(class_share.round(2).rename_axis("Rating"))

Удалено точных повторов текста: 24
Осталось строк: 59,976
Дополнительных совпадений после нормализации: 1


Обучение: 47,980; валидация: 11,996
Пересечений нормализованных текстов: 0


,"обучение, %","валидация, %"
Rating,,
1,31.11,31.11
2,2.72,2.72
3,2.80,2.80
4,5.58,5.59
5,57.79,57.79


## Наивный ориентир

DummyRegressor со стратегией median предсказывает медиану оценок обучающей части для каждого отзыва. Для MAE это разумный постоянный прогноз, не использующий текст.

In [3]:
y_train = train_part["Rating"].to_numpy()
y_valid = valid_part["Rating"].to_numpy()
dummy = DummyRegressor(strategy="median")
dummy.fit(np.zeros((len(train_part), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(valid_part), 1)))
dummy_mae = mean_absolute_error(y_valid, dummy_pred)
print(f"Постоянный прогноз: {dummy_pred[0]:.0f}")
print(f"MAE: {dummy_mae:.5f}")

Постоянный прогноз: 5
MAE: 1.43781


## TF-IDF и логистическая регрессия

TF-IDF учитывает слова и пары слов; словарь обучается только на обучающей части. Стоп-слова не удаляем, чтобы сохранить отрицания. Логистическая регрессия выдаёт вероятности оценок 1–5. В качестве ответа берём первую оценку, на которой накопленная вероятность достигает 0,5: медиану распределения. Такой выбор соответствует метрике MAE. Конфигурацию зафиксировали по предварительному исследованию.

In [4]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_features=150_000,
        sublinear_tf=True,
        dtype=np.float32,
    )),
    ("classifier", LogisticRegression(C=4, max_iter=250, solver="lbfgs")),
])
model.fit(train_part["Review"], y_train)
probabilities = model.predict_proba(valid_part["Review"])
classes = model.named_steps["classifier"].classes_
model_pred = classes[np.argmax(np.cumsum(probabilities, axis=1) >= 0.5, axis=1)]
model_mae = mean_absolute_error(y_valid, model_pred)
print(f"Число признаков TF-IDF: {len(model.named_steps['tfidf'].vocabulary_):,}")
print(f"MAE: {model_mae:.5f}")

Число признаков TF-IDF: 117,969
MAE: 0.19165


## Результаты

Обе модели проверены на одних и тех же отзывах. TF-IDF с логистической регрессией становится простой текстовой базой для дальнейшего сравнения с MPNet и DeBERTa. Отдельно смотрим ошибки редких оценок 2–4, которые общая MAE может скрывать.

In [5]:
results = pd.DataFrame({
    "модель": ["Медиана обучающих оценок", "TF-IDF + Logistic Regression"],
    "MAE": [dummy_mae, model_mae],
})
results["снижение MAE к наивному ориентиру, %"] = (
    100 * (dummy_mae - results["MAE"]) / dummy_mae
)
display(results.round(5))
errors_by_rating = pd.DataFrame({
    "строк в валидации": [int((y_valid == rating).sum()) for rating in classes],
    "MAE текстовой модели": [
        mean_absolute_error(y_valid[y_valid == rating], model_pred[y_valid == rating])
        for rating in classes
    ],
}, index=pd.Index(classes, name="Rating"))
display(errors_by_rating.round(4))

,модель,MAE,"снижение MAE к наивному ориентиру, %"
0,Медиана обучающих оценок,1.43781,0.00000
1,TF-IDF + Logistic Regression,0.19165,86.67092


,строк в валидации,MAE текстовой модели
Rating,,
1,3732,0.1294
2,326,1.0000
3,336,1.2976
4,670,0.8940
5,6932,0.0656


## Выводы

- Наивный прогноз медианы обучающих оценок (5) дал MAE 1,43781. TF-IDF с логистической регрессией и медианой вероятностей дал MAE 0,19165 на той же валидационной части.
- Общая ошибка низкая, но для редких оценок 2–4 она намного выше, чем для 1 и 5. На следующем этапе проверим, помогают ли признаки длины и пунктуации из EDA.
- Это результат одного зафиксированного разбиения. При дальнейшем подборе моделей не будем считать улучшение на этой же выборке независимой итоговой проверкой.